In [1]:
# make_hrv_tables_from_stage1.py
# Combine per-file HRV CSVs (from Stage 1) into group summaries

from pathlib import Path
import pandas as pd
import numpy as np

# ---- Paths ----
PROC = Path("D:/Git_repository/result folder/data/processed")
HRV_ROOT = PROC / "hrv"
SUM_DIR  = PROC / "summaries"
SUM_DIR.mkdir(parents=True, exist_ok=True)

# ---- Groups ----
GROUPS = ["conventional", "vr_high", "vr_low", "stress"]

# ---- Collect all participant HRV CSVs ----
rows = []
for g in GROUPS:
    gdir = HRV_ROOT / g
    if not gdir.exists():
        print(f"⚠️ Skip {g}: folder not found")
        continue
    for f in gdir.glob("*.csv"):
        try:
            h = pd.read_csv(f)
        except Exception as e:
            print(f"⚠️ Failed {f.name}: {e}")
            continue
        if h.empty:
            continue
        h["participant"] = f.stem
        h["group"] = g
        rows.append(h)

if not rows:
    raise SystemExit("❌ No HRV CSVs found — check processed/hrv/* folders.")

# ---- Merge into one table ----
hrv_all = pd.concat(rows, ignore_index=True)
hrv_all.to_csv(SUM_DIR / "hrv_all_participants.csv", index=False)
print("✅ Saved:", SUM_DIR / "hrv_all_participants.csv")

# ---- Compute group mean / std ----
num_cols = hrv_all.select_dtypes(include=[np.number]).columns.tolist()

grp_mean = hrv_all.groupby("group")[num_cols].mean().reset_index()
grp_std  = hrv_all.groupby("group")[num_cols].std(ddof=1).reset_index()

grp_mean.to_csv(SUM_DIR / "hrv_group_mean.csv", index=False)
grp_std.to_csv(SUM_DIR / "hrv_group_std.csv", index=False)

print("✅ Saved:", SUM_DIR / "hrv_group_mean.csv")
print("✅ Saved:", SUM_DIR / "hrv_group_std.csv")

print("\nDone! You can now open these in 'view_hrv_results.py' to visualize.")


✅ Saved: D:\Git_repository\result folder\data\processed\summaries\hrv_all_participants.csv
✅ Saved: D:\Git_repository\result folder\data\processed\summaries\hrv_group_mean.csv
✅ Saved: D:\Git_repository\result folder\data\processed\summaries\hrv_group_std.csv

Done! You can now open these in 'view_hrv_results.py' to visualize.
